# 🏗️ ConvNeXt-Tiny Siamese — Building Damage Assessment (EBD)
### Dedicated Training & Evaluation Pipeline for Kaggle Cloud GPUs
**Honors Project (Session 2026-2027)** | *Shri Ramdeobaba College of Engineering and Management, Nagpur*

---
### 🔍 Why Previous Runs Showed Unrealistic Scores (`0.9987` F1 / `1.0000` Loc_F1):
1. **The Mask Fallback Bug (Root Cause)**: If mask filenames did not match the pattern `pre_disaster_target`, the dataset silently fell back to reading the pre-disaster RGB image as the mask! The model was literally asked to segment high-brightness pixels of the input image (`loc_mask = image > 127`), which a 28M parameter ConvNeXt trivially fits to 1.0000 F1.
2. **Overlapping Tile Leakage**: Random 80/20 splitting placed overlapping 512x512 tiles of the exact same satellite pass in both train and validation.
3. **Zero Augmentation**: The model memorized fixed pixel grids instead of learning damage invariants.

### 🛠️ What Is Fixed in This Notebook:
- **Strict Mask Verification**: Never uses the satellite image as a mask; validates ground truth labels.
- **Disaster-Event-Aware Split**: Train and Validation use images from **completely different natural disaster events** (zero data leakage).
- **Full Albumentations Pipeline**: Flips, 90° rotations, scale-shift-rotate, color jitter, cutout, and noise.
- **Realistic Benchmark Target**: Overall Score ~`0.70 - 0.76` (matching official xView2 / EBD state of the art).

---
### ⚡ How to Train Offline (Shut Down Laptop & Let Kaggle Train):
1. Under **Notebook options** (right sidebar) → **Accelerator**: Select **GPU P100** or **GPU T4 x2**.
2. Under **Notebook options** → **Internet**: Toggle to **ON**.
3. Click **+ Add Input** → select your **EBD dataset**.
4. Click the blue **Save Version** (top right) → Choose **Save & Run All (Commit)** → Click **Save**.
5. **You can now close your browser, turn off Wi-Fi, and go offline!** Kaggle runs the training in the cloud and saves `ConvNeXt_Siamese_best.pth` in the **Output** tab.

In [ ]:
# ── Cell 1: Check GPU Accelerator & Environment ──────────────────────────────
!nvidia-smi

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# ── Cell 2: Install Dependencies ─────────────────────────────────────────────
!pip install -q albumentations opencv-python matplotlib scikit-learn pillow torchvision tqdm

In [ ]:
# ── Cell 3: Setup Project Modules in /kaggle/working ─────────────────────────
import os, sys, shutil, zipfile, io, base64

KAGGLE_WORKING = '/kaggle/working'
os.chdir(KAGGLE_WORKING)

# Check if honors_code.zip or python files exist in /kaggle/input
code_found = False
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            src = os.path.join(root, f)
            if f == 'honors_code.zip':
                print(f'📦 Found {f}! Unpacking into {KAGGLE_WORKING} ...')
                with zipfile.ZipFile(src, 'r') as z:
                    z.extractall(KAGGLE_WORKING)
                code_found = True
                break
            elif f.endswith('.py') and not f.startswith('.'):
                dest = os.path.join(KAGGLE_WORKING, 'models' if 'models' in root else '', f)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy(src, dest)
                code_found = True

# Automatic fallback: If no code zip was attached, unpack built-in project modules
if not code_found and not os.path.exists(os.path.join(KAGGLE_WORKING, 'train.py')):
    print('📦 Unpacking built-in project modules ...')
    EMBEDDED_ZIP_B64 = '...'
    zip_bytes = base64.b64decode(EMBEDDED_ZIP_B64)
    with zipfile.ZipFile(io.BytesIO(zip_bytes), 'r') as z:
        z.extractall(KAGGLE_WORKING)
    print('✅ Built-in project modules extracted successfully!')
else:
    print('✅ Project code ready in /kaggle/working!')

if KAGGLE_WORKING not in sys.path:
    sys.path.insert(0, KAGGLE_WORKING)

print('Working directory:', os.getcwd())
print('sys.path[0]:', sys.path[0])


In [ ]:
# ── Cell 4: Smart EBD Dataset Detection & Strict Mask Validation ─────────────
import os, glob, zipfile, tarfile

KAGGLE_INPUT = "/kaggle/input"
LOCAL_DATA_DIR = "/kaggle/working/data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

found_images = 0
data_path_to_use = None

if os.path.exists(KAGGLE_INPUT):
    # Check archives first
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        for f in files:
            fpath = os.path.join(root, f)
            if f.endswith(".zip") and any(k in f.lower() for k in ["ebd", "dataset", "damage", "xbd"]):
                if len(os.listdir(LOCAL_DATA_DIR)) == 0:
                    print(f"📦 Extracting {f} into {LOCAL_DATA_DIR} ...")
                    with zipfile.ZipFile(fpath, 'r') as z:
                        z.extractall(LOCAL_DATA_DIR)
                    print("✅ Extraction complete!")
                data_path_to_use = LOCAL_DATA_DIR
                break
            elif (f.endswith(".tar.gz") or f.endswith(".tgz")) and any(k in f.lower() for k in ["ebd", "dataset", "damage", "xbd"]):
                if len(os.listdir(LOCAL_DATA_DIR)) == 0:
                    print(f"📦 Extracting {f} into {LOCAL_DATA_DIR} ...")
                    with tarfile.open(fpath, 'r:gz') as t:
                        t.extractall(LOCAL_DATA_DIR)
                    print("✅ Extraction complete!")
                data_path_to_use = LOCAL_DATA_DIR
                break

    # If no archive, scan for extracted image folders
    if not data_path_to_use:
        for root, dirs, files in os.walk(KAGGLE_INPUT):
            img_count = sum(1 for f in files if "pre_disaster" in f and not any(k in f for k in ["target", "mask", "damage", "loc"]))
            if img_count > 0:
                found_images += img_count
                data_path_to_use = root
                print(f"✅ Found {img_count} pre-disaster images directly in {root}")
                break

if not data_path_to_use or found_images == 0:
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        img_count = sum(1 for f in files if "pre_disaster" in f and not any(k in f for k in ["target", "mask", "damage", "loc"]))
        if img_count > 0:
            found_images += img_count
            data_path_to_use = root
            break

print("----------------------------------------------------------------------")
if data_path_to_use and found_images > 0:
    print(f"🎯 Active Dataset Directory: {data_path_to_use}")
    print(f"📊 Total Discovered Bi-temporal Image Pairs: {found_images}")
else:
    data_path_to_use = LOCAL_DATA_DIR
    print("⚠️ No real EBD dataset found in /kaggle/input yet.")
    print("   Please add the EBD dataset via '+ Add Input' in the right sidebar.")
print("----------------------------------------------------------------------")

In [ ]:
# ── Cell 5: Setup Output Folders & Flush GPU Cache ───────────────────────────
import gc, torch
from config import CHECKPOINT_DIR, VISUALS_DIR, OUTPUT_DIR

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(VISUALS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Outputs Directory: {OUTPUT_DIR}")
print(f"Checkpoints:       {CHECKPOINT_DIR}")
print(f"Visualizations:    {VISUALS_DIR}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("✅ GPU Memory Cleared. Ready to train ConvNeXt-Tiny!")

In [ ]:
# ── Cell 6: Train ConvNeXt-Tiny Siamese BDA (Dedicated Training) ─────────────
# Features:
#   - Only ConvNeXt-Tiny Siamese (avoids spending hours on ResNeXt / Hybrid)
#   - Albumentations data augmentation active
#   - Disaster-event-aware split prevents leakage
#   - Early stopping (patience=5)
#   - Real-time logging of F1, Precision, Recall, and Overall EBD score

from train import train_convnext_only

# 20 Epochs with Early Stopping gives optimal convergence on real EBD data
EPOCHS = 20

results = train_convnext_only(
    data_dir=data_path_to_use,
    epochs=EPOCHS,
    patience=5
)

In [ ]:
# ── Cell 7: Plot Training Curves (Loss, IoU, Overall F1) ──────────────────────
import json, matplotlib.pyplot as plt
from IPython.display import Image, display

hist_file = os.path.join(OUTPUT_DIR, "ConvNeXt_Siamese_history.json")
if os.path.exists(hist_file):
    with open(hist_file, "r") as f:
        hist = json.load(f)

    epochs_ran = range(1, len(hist["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss curve
    ax1.plot(epochs_ran, hist["train_loss"], 'b-o', label="Training Loss", linewidth=2)
    ax1.set_title("ConvNeXt-Tiny Siamese Training Loss", fontsize=13, fontweight="bold")
    ax1.set_xlabel("Epoch", fontsize=11)
    ax1.set_ylabel("Loss", fontsize=11)
    ax1.grid(True, linestyle="--", alpha=0.6)
    ax1.legend()

    # Validation Metrics
    ax2.plot(epochs_ran, hist["val_score"], 'g-s', label="Overall EBD Score", linewidth=2)
    ax2.plot(epochs_ran, hist["val_f1_loc"], 'c-^', label="Localization F1", linewidth=2)
    ax2.plot(epochs_ran, hist["val_f1_damage"], 'r-d', label="Harmonic Damage F1", linewidth=2)
    ax2.set_title("Validation Performance over Epochs", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Epoch", fontsize=11)
    ax2.set_ylabel("Score / F1", fontsize=11)
    ax2.grid(True, linestyle="--", alpha=0.6)
    ax2.legend()

    curve_path = os.path.join(VISUALS_DIR, "convnext_training_curves.png")
    plt.tight_layout()
    plt.savefig(curve_path, dpi=200)
    plt.close()
    display(Image(curve_path))
    print(f"✅ Saved training curves to: {curve_path}")

In [ ]:
# ── Cell 8: Generate 6-Panel Damage Overlays on Real Disaster Samples ─────────
from evaluate import infer_and_evaluate, find_checkpoint
from IPython.display import Image, display

conv_ckpt = find_checkpoint(model_name="ConvNeXt_Siamese")
if conv_ckpt:
    print(f"🔍 Evaluating ConvNeXt-Tiny Checkpoint: {conv_ckpt}")
    infer_and_evaluate(conv_ckpt, model_name="ConvNeXt_Siamese", num_samples=5)
    
    for s_idx in [1, 2]:
        vis_path = os.path.join(VISUALS_DIR, f"ConvNeXt_Siamese_prediction_sample_{s_idx}.png")
        if os.path.exists(vis_path):
            print(f"Displaying Sample {s_idx}:")
            display(Image(vis_path))
else:
    print("❌ Checkpoint not found. Run Cell 6 first!")

In [ ]:
# ── Cell 9: Bundle Checkpoint & Visualizations for 1-Click Download ───────────
import zipfile, os

zip_filename = "/kaggle/working/convnext_bda_outputs.zip"
print(f"📦 Creating {zip_filename} containing ConvNeXt checkpoint and visualizations...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for f in files:
            full_f = os.path.join(root, f)
            rel_f = os.path.relpath(full_f, "/kaggle/working")
            z.write(full_f, rel_f)

print(f"✅ Successfully packaged all outputs ({os.path.getsize(zip_filename) / (1024*1024):.2f} MB)!")
print("\n📁 Downloadable files ready in Kaggle Output tab:")
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        p = os.path.join(root, f)
        if not any(k in p for k in [".git", "__pycache__"]):
            print(f"  • {os.path.relpath(p, '/kaggle/working')} ({os.path.getsize(p)/(1024*1024):.2f} MB)")